# Upload a large fine-tuning dataset in parts

Companion to the blog post *Making Room for Bigger Fine-Tuning Datasets*. Build a 1.2 GB chat dataset, upload it to Crusoe in 128 MiB parts in parallel, complete the upload, and poll until the assembled file is ready for a fine-tuning job. Everything goes through the OpenAI-compatible [Uploads API](https://docs.crusoecloud.com/api/managed-ai/#tag/Uploads); no GPU is involved.

<div align="center"><img src="assets/multipart-flow.png" alt="A training file split into parts on your machine, sent in parallel into an upload session on Crusoe, completed with ordered part ids and a checksum, and assembled into one file id" width="900"></div>

## Setup

Paste your key into `CRUSOE_API_KEY`. Create one in the Crusoe Console under [Inference API keys](https://console.crusoecloud.com/security/inference-api-keys). Helpers for the dataset export, part reading, parallel sends, and polling live in [utils.py](utils.py).

In [1]:
import os
from pathlib import Path

import requests
from openai import OpenAI  # the Uploads API needs openai >= 1.37

import utils

CRUSOE_API_KEY = os.environ.get("CRUSOE_API_KEY", "paste-your-crusoe-api-key-here")
HF_TOKEN = None  # optional Hugging Face token, only for gated datasets

if "paste-your" in CRUSOE_API_KEY:
    raise ValueError("Add your Crusoe API key to CRUSOE_API_KEY above")

BASE_URL = "https://api.intelligence.crusoecloud.com/v1"
HEADERS = {"Authorization": f"Bearer {CRUSOE_API_KEY}"}
client = OpenAI(api_key=CRUSOE_API_KEY, base_url=BASE_URL)

MiB = 1024 * 1024
PART_SIZE = 128 * MiB  # Crusoe's ceiling per part
WORKERS = 8            # parts in flight at once; peak RAM is about WORKERS * PART_SIZE

## Step 1: Build a large training file

The `train_sft` split of [HuggingFaceH4/ultrachat_200k](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) is about 208,000 conversations already in chat format. Set `MAX_ROWS` to an integer for a smaller trial file.

In [2]:
MAX_ROWS = None  # for example 20_000 rows for a quick run of about 115 MB

TRAIN_PATH = utils.build_training_file(
    "HuggingFaceH4/ultrachat_200k", "train_sft", "data/train.jsonl", max_rows=MAX_ROWS, token=HF_TOKEN
)
FILE_BYTES = TRAIN_PATH.stat().st_size
N_PARTS = utils.part_count(TRAIN_PATH, PART_SIZE)
print(f"{FILE_BYTES / MiB:,.0f} MiB on disk, {N_PARTS} parts of up to {PART_SIZE // MiB} MiB")

1,185 MiB on disk, 10 parts of up to 128 MiB


## Step 2: Create the upload

Declare the filename, purpose, size, and MIME type. The bytes received across all parts must add up to this size at completion. The session stays open for two hours.

In [3]:
upload = client.uploads.create(
    purpose="fine-tune",
    filename=TRAIN_PATH.name,
    bytes=FILE_BYTES,
    mime_type="text/jsonl",
)
print(f"{upload.id}  status={upload.status}  bytes={upload.bytes:,}  session={(upload.expires_at - upload.created_at) // 3600}h")

upload_bf8f98c4d6584ba7bc9b8b398c576f8d  status=pending  bytes=1,242,536,347  session=2h


## Step 3: Add the parts in parallel

Each part is an independent request that returns a part id. Parts can go in any order and at the same time; a failed part is re-sent on its own and every retry mints a fresh id.

In [4]:
def send_part(index):
    data = utils.read_part(TRAIN_PATH, index, PART_SIZE)
    part = client.uploads.parts.create(upload_id=upload.id, data=data)
    return index, part.id


part_ids, elapsed = utils.send_in_parallel(send_part, N_PARTS, WORKERS)
print(f"\n{FILE_BYTES / MiB:,.0f} MiB in {elapsed:.0f} s with {WORKERS} workers, about {FILE_BYTES / MiB / elapsed:.0f} MiB/s")

[ 24.7s] part  1/10 landed  part_01918a79f013466a8b7e023f05739b03


[ 26.1s] part  5/10 landed  part_dab6d6bafe984f82a9aae11e4f010936
[ 26.2s] part  4/10 landed  part_fc224d143d8e4d5a94178ff0b3cc0ad5
[ 26.2s] part  8/10 landed  part_ce8a47db307c416cb9faf8425b2966f2
[ 26.3s] part  7/10 landed  part_2cb23b04590e46b5b2074eaa25de269a


[ 26.8s] part  3/10 landed  part_167d723dac9f4d5fb44301123b80730c
[ 26.8s] part  2/10 landed  part_d328af6ca4104f099c624e98e4566a21
[ 26.8s] part  6/10 landed  part_54fd728cc5a84340be90d846c33d51ad


[ 30.0s] part 10/10 landed  part_85d2278fe8e64b7db445c639f711116d


[ 31.9s] part  9/10 landed  part_74e33e4923524914b9b55d3747eca700

1,185 MiB in 32 s with 8 workers, about 37 MiB/s


Two REST endpoints show an upload from the outside: the parts Crusoe has received for a session, and the sessions still open.

In [5]:
received = requests.get(f"{BASE_URL}/uploads/{upload.id}/parts", headers=HEADERS).json()
print(f"{len(received['data'])} of {N_PARTS} parts received")

pending = requests.get(f"{BASE_URL}/uploads", headers=HEADERS, params={"status": "pending"}).json()
for session in pending["data"]:
    print(f"pending: {session['id']}  {session['filename']}  {session['bytes']:,} bytes")

10 of 10 parts received


pending: upload_bf8f98c4d6584ba7bc9b8b398c576f8d  train.jsonl  1,242,536,347 bytes


## Step 4: Complete the upload

Pass the part ids in file order and, optionally, the file's MD5 so Crusoe verifies the assembled file. Unlike OpenAI, the call returns `pending` right away and assembly runs in the background.

In [6]:
FILE_MD5 = utils.md5_of(TRAIN_PATH)
completion = client.uploads.complete(upload_id=upload.id, part_ids=part_ids, md5=FILE_MD5)
print(f"md5={FILE_MD5}  status={completion.status}")

md5=c331df73b21e3752cfea8758b11f26f4  status=pending


## Step 5: Poll until the file is ready

Retrieve the upload until its status leaves `pending`. On `completed` the `file` field holds a regular File object; on `failed` the `error` field carries a code such as `checksum_mismatch` or `missing_part`.

In [7]:
state, waited = utils.wait_for_assembly(BASE_URL, HEADERS, upload.id)
if state["status"] != "completed":
    raise RuntimeError(f"assembly {state['status']}: {state.get('error')}")

TRAINING_FILE_ID = state["file"]["id"]
print(f"assembled in {waited:.0f} s")
print(f"file id: {TRAINING_FILE_ID}")

training_file = client.files.retrieve(TRAINING_FILE_ID)
print(f"{training_file.filename}  {training_file.bytes:,} bytes  purpose={training_file.purpose}  status={training_file.status}")

assembled in 17 s
file id: files:file_bf8f98c4d6584ba7bc9b8b398c576f8d:b2b31910-9093-4b1b-8aa4-2fce722b9898:1f6469aba3fd01adad504547fc93b1186a9e51fd


train.jsonl  1,242,536,347 bytes  purpose=fine-tune  status=uploaded


## Step 6: Use the file in a fine-tuning job

The file id goes wherever a file id goes today. The job is off by default because a run over 208,000 conversations is billed per training token.

In [8]:
SUBMIT_FINE_TUNING_JOB = False
BASE_MODEL_NAME = "Qwen/Qwen3-8B"

if SUBMIT_FINE_TUNING_JOB:
    fine_tunable = [m for m in client.models.list().data if getattr(m, "fine_tuning_available", False)]
    base_model_id = next(m.id for m in fine_tunable if m.model_name == BASE_MODEL_NAME)
    job = client.fine_tuning.jobs.create(model=base_model_id, training_file=TRAINING_FILE_ID, suffix="ultrachat-multipart")
    print(f"{job.id}  status={job.status}")
    print(f"https://console.crusoecloud.com/foundry/fine-tuning/jobs/{job.id}/overview")
else:
    print("Skipped. Set SUBMIT_FINE_TUNING_JOB = True to train on the uploaded file.")

Skipped. Set SUBMIT_FINE_TUNING_JOB = True to train on the uploaded file.


## Step 7: Cancel and clean up

Cancelling a session releases its parts at once; an abandoned session expires after two hours. The assembled file persists until you delete it.

In [9]:
abandoned = client.uploads.create(purpose="fine-tune", filename="abandoned.jsonl", bytes=FILE_BYTES, mime_type="text/jsonl")
print(f"{abandoned.id}  status={client.uploads.cancel(abandoned.id).status}")

upload_491156738e94453293a702615d863e63  status=cancelled


In [10]:
DELETE_UPLOADED_FILE = False

if DELETE_UPLOADED_FILE:
    print(client.files.delete(TRAINING_FILE_ID))
else:
    print("Skipped. Set DELETE_UPLOADED_FILE = True to delete the uploaded file.")

Skipped. Set DELETE_UPLOADED_FILE = True to delete the uploaded file.
